# REFLECT – End-to-End Question Generation Eval
Retrieves top-k journal chunks per query, generates a clarifying question, then scores rule compliance and LLM-judge quality.

## Config

In [ ]:
from pathlib import Path
import re, json
from statistics import mean, pstdev
from typing import Any
import pandas as pd
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

SCRIPT_DIR      = Path(".").resolve()
EVAL_DATA_PATH  = SCRIPT_DIR / "eval_data"
RESULTS_PATH    = SCRIPT_DIR / "eval_results"
OLLAMA_BASE_URL = "http://localhost:57703"
EMBED_MODEL     = "nomic-embed-text"

# ⚠️  'gpt-oss:20b' is a placeholder – replace with your actual Ollama tag.
# ⚠️  Avoid using the same models as both generator AND judge (self-grading bias).
MODELS_TO_TEST  = ["mistral", "llama3", "gpt-oss:20b"]
JUDGE_MODELS    = ["mistral", "llama3"]

TOP_K           = 3
MAX_QUERIES     = None   # e.g. 2 for a quick smoke test
REQUEST_TIMEOUT = 600.0

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
print("Config OK")


Config OK


## Prompts

In [7]:
REFLECT_PROMPT = """You are a reflective journaling assistant.
Based on the following excerpts from a user's journal, generate a single
clarifying question that helps the writer go deeper on a theme that appears
across their entries.

Rules:
- Ask about ONE specific theme from the excerpts
- Do not ask yes/no questions
- Do not start with "How does that make you feel"
- Keep it under 20 words

Journal excerpts:
{context}

Question:"""

JUDGE_PROMPT = """You are evaluating a journaling question generated by an AI assistant.
Respond only with valid JSON, no extra text.

Journal excerpts used as context:
{context}

Generated question:
{question}

Rate each dimension 1-5:
- specificity: references concrete themes from these excerpts.
- depth: likely to push meaningful reflection.
- context_grounding: clearly anchored in retrieved excerpts, not generic.

{{"specificity": N, "depth": N, "context_grounding": N, "reason": "one sentence"}}"""

TEST_QUERIES = [
    "What recurring patterns appear in how I handle work stress?",
    "What does my relationship with productivity look like?",
    "How do I talk about my relationships with other people?",
    "What themes come up when I am having a low energy day?",
    "What am I avoiding or procrastinating on?",
]
print("Prompts loaded")


Prompts loaded


## Helpers

In [8]:
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['-][A-Za-z0-9]+)?")
YES_NO_STARTERS = {
    "is","are","am","was","were","do","does","did","can","could",
    "would","will","have","has","had","should","shall","may","might","must",
}
BANNED_OPENER = "how does that make you feel"

def build_llm(model_name: str) -> Ollama:
    extra = {"think": False} if "qwen" in model_name.lower() else {}
    return Ollama(model=model_name, base_url=OLLAMA_BASE_URL,
                  request_timeout=REQUEST_TIMEOUT, additional_kwargs=extra)

def extract_json_dict(raw: str) -> dict:
    text = raw.strip().lstrip("```").lstrip("json").strip().rstrip("```").strip()
    try:
        p = json.loads(text); return p if isinstance(p, dict) else {}
    except json.JSONDecodeError:
        s, e = text.find("{"), text.rfind("}") + 1
        if s == -1 or e <= s: return {}
        try: p = json.loads(text[s:e]); return p if isinstance(p, dict) else {}
        except: return {}

def to_score(v) -> int | None:
    try: s = int(v)
    except: return None
    return max(1, min(5, s))

def analyze_rules(question: str) -> dict:
    clean = " ".join(question.strip().strip('"').split())
    words = WORD_RE.findall(clean)
    wc    = len(words)
    first = words[0].lower() if words else ""
    yn    = first in YES_NO_STARTERS
    ban   = clean.lower().startswith(BANNED_OPENER)
    v     = int(wc > 20) + int(yn) + int(ban)
    s     = 5 - (2 if wc > 20 else 0) - (2 if yn else 0) - (3 if ban else 0)
    return {"word_count": wc, "is_yes_no": yn, "starts_with_banned": ban,
            "strict_rule_pass": v == 0, "rule_following_det": max(1, s)}

def generate_question(llm, context: str) -> str:
    return str(llm.complete(REFLECT_PROMPT.format(context=context))).strip()

def judge_question(judge_llm, context: str, question: str) -> dict:
    raw = str(judge_llm.complete(JUDGE_PROMPT.format(
        context=context, question=question))).strip()
    return extract_json_dict(raw)

def aggregate_judgements(payloads: dict) -> dict:
    metrics = ("specificity", "depth", "context_grounding")
    per = {}; reasons = []; valid = 0
    for jname, payload in payloads.items():
        row = {m: to_score(payload.get(m)) for m in metrics}
        per[jname] = row
        if any(v is not None for v in row.values()): valid += 1
        r = payload.get("reason","")
        if isinstance(r, str) and r.strip(): reasons.append(f"{jname}: {r.strip()}")
    agg = {}
    for m in metrics:
        vals = [row[m] for row in per.values() if row[m] is not None]
        agg[m] = round(mean(vals), 3) if vals else None
        agg[f"{m}_judge_std"] = (None if not vals else 0.0 if len(vals)==1
                                 else round(pstdev(vals), 3))
    n = max(1, len(payloads))
    agg["judge_valid_count"] = valid
    agg["judge_valid_rate"]  = round(valid / n, 3)
    agg["judge_reasons"]     = " | ".join(reasons)
    agg["judge_scores_json"] = json.dumps(per, ensure_ascii=True)
    return agg

print("Helpers loaded")


Helpers loaded


## Build Index
> Tip: persist the index to disk with `StorageContext` to avoid re-embedding on every run.

In [9]:
documents = SimpleDirectoryReader(str(EVAL_DATA_PATH)).load_data()
print(f"Loaded {len(documents)} document(s)")

embed = OllamaEmbedding(model_name=EMBED_MODEL, base_url=OLLAMA_BASE_URL)
index = VectorStoreIndex.from_documents(documents, embed_model=embed)
print("Index built")


Loaded 15 document(s)
Index built


## Run Eval

In [11]:
queries    = TEST_QUERIES[:MAX_QUERIES] if MAX_QUERIES else TEST_QUERIES
judge_llms = {j: build_llm(j) for j in JUDGE_MODELS}

print(f"Queries: {len(queries)}  |  Generators: {len(MODELS_TO_TEST)}  |  Judges: {list(judge_llms)}")
rows = []

for model_name in MODELS_TO_TEST:
    print(f"\n── {model_name} ──")
    gen = build_llm(model_name)
    for query in queries:
        retriever = index.as_retriever(similarity_top_k=TOP_K)
        nodes     = retriever.retrieve(query)
        chunks    = [n.get_content() for n in nodes]
        context   = "\n\n---\n\n".join(chunks)
        question  = generate_question(gen, context)
        rule_stats = analyze_rules(question)
        payloads   = {j: judge_question(jllm, context, question)
                      for j, jllm in judge_llms.items()}
        agg = aggregate_judgements(payloads)
        print(f"  Q: {query[:65]}...")
        print(f"  A: {question}")
        rows.append({"model": model_name, "query": query,
                     "retrieved_chunks": len(chunks),
                     "question": question, **rule_stats, **agg})

df = pd.DataFrame(rows)
print("\nDone.")


Queries: 5  |  Generators: 3  |  Judges: ['mistral', 'llama3']

── mistral ──


ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

## Summary

In [ ]:
summary = (
    df.groupby("model")
    .agg(samples=("question","count"),
         specificity=("specificity","mean"), depth=("depth","mean"),
         context_grounding=("context_grounding","mean"),
         rule_following_det=("rule_following_det","mean"),
         strict_rule_pass_rate=("strict_rule_pass","mean"),
         judge_valid_rate=("judge_valid_rate","mean"))
    .round(3)
    .sort_values(["specificity","depth","context_grounding","rule_following_det"],
                 ascending=False)
)

per_path = RESULTS_PATH / "e2e_eval.csv"
sum_path = RESULTS_PATH / "e2e_eval_summary.csv"
df.to_csv(per_path, index=False)
summary.to_csv(sum_path)

print(summary)
print(f"\nSaved: {per_path}\n       {sum_path}")
